# Factorized (2+1)D Glow and per-axis squeezing
Treat the first spatial axis as time: `(batch, channels, time, height, width)`. `Squeeze3d((1, 2, 2))` preserves time resolution. Per-axis kernels in `ConvNet3d`, exposed through `GlowBlock3d`, let the conditioner learn spatial and temporal structure separately.

## Setup
Install the current checkout with `python -m pip install -e '.[examples]'` from the repository root. These examples exercise recent source APIs, which may not yet be in an older installed release. Select that environment's kernel and run all cells in order.

All observations are synthetic; no download is required. The small training budgets demonstrate the API rather than a converged scientific model. Increase `steps` and model capacity for a longer experiment.

In [ ]:
import torch
import antsnormflows as nf
from matplotlib import pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Package:", nf.__version__, "Device:", device)


## Space-to-channel packing without temporal downsampling
The shape changes from `(B, 1, 5, 8, 8)` to `(B, 4, 5, 4, 4)`. Only height and width must be divisible by two; the odd time length is allowed. Forward reverses this packing and both log-determinants are zero.

In [ ]:
shape = (1, 5, 8, 8)
factors = (1, 2, 2)
squeeze = nf.flows.Squeeze3d(factors)
x = torch.randn(2, *shape, device=device)
packed, inverse_ld = squeeze.inverse(x)
reconstructed, forward_ld = squeeze(packed)
assert packed.shape == (2, 4, 5, 4, 4)
torch.testing.assert_close(reconstructed, x, atol=0, rtol=0)
print("Data:", tuple(x.shape), "Packed:", tuple(packed.shape))


## Compare spatial-only, factorized, and isotropic conditioners
The three kernel entries correspond to successive convolutions. Factorized kernels use spatial `(1,3,3)`, temporal `(3,1,1)`, then spatial `(1,3,3)` convolutions. With `temporal_init="identity"`, the middle convolution starts as a per-channel identity. The zero-initialized final convolution still makes the initial coupling an identity; temporal dependence is learned during training.

A spatial-only conditioner does not mix frames after normalization has been initialized. This describes the conditioner, not a guarantee of identical marginal distributions across time.

In [ ]:
kernels = {
    "spatial_only": ((1, 3, 3), (1, 1, 1), (1, 3, 3)),
    "factorized": ((1, 3, 3), (3, 1, 1), (1, 3, 3)),
    "isotropic": (3, 1, 3),
}
def make_model(kind):
    block = nf.flows.GlowBlock3d(
        4, hidden_channels=8, kernel_size=kernels[kind],
        temporal_init="identity" if kind == "factorized" else "default",
        net_actnorm=False, s_cap=0.5, actnorm_s_cap=5.0,
        conv_s_cap=1.0, shift_cap=3.0,
    )
    return nf.NormalizingFlow(
        nf.distributions.DiagGaussian((4, 5, 4, 4)),
        [block, nf.flows.Squeeze3d(factors)],
    ).to(device)

models = {kind: make_model(kind) for kind in kernels}
for kind, model in models.items():
    print(kind, "parameters:", sum(p.numel() for p in model.parameters()))
# Inspect the temporal identity initialization before training.
convs = [m for m in models["factorized"].modules() if isinstance(m, torch.nn.Conv3d)]
middle = convs[1]
expected = torch.zeros_like(middle.weight)
for channel in range(8):
    expected[channel, channel, 1, 0, 0] = 1
torch.testing.assert_close(middle.weight, expected)


In [ ]:
axis = torch.linspace(-1, 1, 8, device=device)
h, w = torch.meshgrid(axis, axis, indexing="ij")
time = torch.linspace(-1, 1, 5, device=device)
def draw_sequences(batch):
    offset = 0.3 * torch.randn(batch, 1, 1, 1, device=device)
    center = offset + 0.4 * time[None, :, None, None]
    blob = torch.exp(-((w[None, None] - center)**2 + h[None, None]**2) / 0.3)
    return blob[:, None] + 0.15 * torch.randn(batch, *shape, device=device)

observations, validation = draw_sequences(16), draw_sequences(16)
steps = 30
histories = {}
for kind, model in models.items():
    with torch.no_grad():
        model.log_prob(observations)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    history = []
    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = model.forward_kld(draw_sequences(8))
        assert torch.isfinite(loss)
        loss.backward()
        optimizer.step()
        history.append(loss.item() / observations[0].numel())
    model.eval()
    with torch.no_grad():
        z, inverse_ld = model.inverse_and_log_det(validation)
        back, forward_ld = model.forward_and_log_det(z)
        torch.testing.assert_close(back, validation, atol=2e-4, rtol=2e-4)
        torch.testing.assert_close(inverse_ld + forward_ld, torch.zeros_like(inverse_ld),
                                   atol=2e-3, rtol=0)
        print(kind, "validation NLL per scalar:",
              -model.log_prob(validation).mean().item() / observations[0].numel())
    histories[kind] = history
for kind, history in histories.items():
    plt.plot(history, label=kind)
plt.xlabel("Step")
plt.ylabel("NLL per scalar (nats)")
plt.legend()
plt.show()


## Inspect generated frames
These short runs exercise the architectures, not a controlled performance comparison. Reliable model selection needs matched training budgets, repeated seeds, and held-out data. In this example `NormalizingFlow` is single-scale, so its `sample` method needs no multiscale shape cache.

In [ ]:
with torch.no_grad():
    samples, log_q = models["factorized"].sample(2)
    torch.testing.assert_close(log_q, models["factorized"].log_prob(samples),
                               atol=3e-3, rtol=2e-4)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for column in range(5):
    axes[0, column].imshow(validation[0, 0, column].cpu(), cmap="gray")
    axes[1, column].imshow(samples[0, 0, column].cpu(), cmap="gray")
    axes[0, column].set_title(f"Frame {column}")
axes[0, 0].set_ylabel("Observed")
axes[1, 0].set_ylabel("Generated")
plt.tight_layout()
plt.show()
